# Exercise 3: Fourier properties

This exercise builds intuition for several DFT properties covered in the lectures, and introduces the `dftModel` module from `smstools`. It has four parts:
1. Minimize energy spread in the DFT of two sinusoids
2. Symmetry properties of the DFT
3. Suppress frequency components using the DFT model
4. FFT size and zero-padding

### Relevant concepts

**DFT of sinusoids:** When a real sinusoid has an integer number of cycles in $N$ samples, its frequency exactly matches one of the DFT bin frequencies and the spectrum has a value of zero at every other bin. Otherwise, the sinusoid's energy spreads over all bins (spectral leakage). For a cosine at bin frequency $k_0$:

$$x[n] = A_0\cos\!\left(\frac{2\pi k_0 n}{N}\right) \quad\Longrightarrow\quad
X[k] = \begin{cases} A_0/2 & k = k_0 \text{ or } k = N-k_0 \\ 0 & \text{otherwise} \end{cases}$$

**Zero-padding:** Appending zeros to a signal before computing its DFT produces an interpolated version of the spectrum of the original signal. Most FFT implementations apply zero-padding implicitly when the requested DFT size exceeds the signal length.

**Zero-phase windowing:** By circularly shifting a frame so that its centre moves to index 0 before the DFT, we avoid the linear phase offset that a time delay would otherwise introduce (a shift multiplies the DFT by a complex exponential, leaving the magnitude intact but rotating the phase). Combined with zero-padding, this produces the standard `fftbuffer`:

```python
hM1 = floor((M+1)/2)
hM2 = floor(M/2)
dftbuffer = np.zeros(N)
dftbuffer[:hM1] = x[hM2:]
dftbuffer[-hM2:] = x[:hM2]
```

**Real, even, and odd signals:** A signal is *even* if $x[n] = x[-n]$ and *odd* if $x[n] = -x[-n]$. For a zero-phase windowed signal of odd length $M$, this means $x[n] = x[M-n]$ (even) or $x[n] = -x[M-n]$ (odd) for $1 \le n \le M-1$. DFT symmetry properties:

- If $x$ is **real**: $|X[k]| = |X[M-k]|$ and $\angle X[k] = -\angle X[M-k]$
- If $x$ is **real and even**: additionally $\mathrm{Im}(X[k]) = 0$ for all $k$

**Positive half of the DFT spectrum:** Because real signals have conjugate-symmetric spectra, storing only the first $(N/2)+1$ bins is sufficient. The magnitude spectrum of the positive half (in dB) is:
$$m_X = 20\log_{10}|X[0:(N/2)+1]|$$

**Filtering via the DFT:** Filtering multiplies the DFT of the input by the DFT of a filter's impulse response — equivalent to circular convolution in the time domain. A simple but illustrative filter can be implemented by zeroing out the DFT bins whose frequencies fall below a desired cutoff.

$$x_1[n] * x_2[n] \;\Longleftrightarrow\; X_1[k]\,X_2[k]$$


## Part 1 – Minimize energy spread in DFT of sinusoids

Given an input signal containing two sinusoids at frequencies `f1` and `f2` (both positive integer factors of `fs`), the function `minimize_energy_spread_dft()` should select the smallest number of samples `M` such that the positive-half DFT magnitude spectrum has **exactly two non-zero values** (one per sinusoid), and return that spectrum.

**How to find M:** The DFT of a sinusoid is zero at every bin except the two that match its frequency, only when the DFT length contains an integer number of sinusoid periods. For two sinusoids, this condition must hold simultaneously, so `M` is the **Least Common Multiple** of the two sinusoid periods (in samples):

$$T_i = f_s / f_i \quad \text{(samples per period)}, \qquad M = \mathrm{lcm}(T_1, T_2) = \frac{T_1 \cdot T_2}{\gcd(T_1, T_2)}$$

You can use `math.gcd` to compute the GCD.

**Inputs:**
- `x` (np.array): signal of length $\ge M$, containing sinusoids at `f1` and `f2`
- `fs` (float): sampling frequency in Hz
- `f1`, `f2` (float): sinusoid frequencies in Hz

**Output:** `mX` — positive half of the DFT magnitude spectrum (in dB), length $(M/2)+1$.

Compute `mX = 20*np.log10(np.abs(X[:M//2+1]))` where `X` is the `M`-point FFT of the first `M` samples of `x`. Due to floating-point precision, values below $-120$ dB can be treated as zero.


In [ ]:
from scipy.fftpack import fft, fftshift
import numpy as np
from math import gcd, ceil, floor
from smstools.models.dftModel import dftAnal, dftSynth
from scipy.signal import get_window
import matplotlib.pyplot as plt

In [ ]:
# E3 - 1.1: Complete the function minimize_energy_spread_dft()

def minimize_energy_spread_dft(x, fs, f1, f2):
    """ From a signal with two sinusoids compute its magnitude spectrum having only two non-zero value.

    Args:
        x (np.array): input signal
        fs (float): sampling frequency in Hz
        f1 (float): frequency of first sinusoid component in Hz
        f2 (float): frequency of second sinusoid component in Hz

    Returns:
        np.array: positive half of magnitude spectrum (in dB)

    """
    ### Your code here



Test cases for `minimize_energy_spread_dft()`:

**Test case 1:** `fs = 10000 Hz`, `f1 = 80 Hz`, `f2 = 200 Hz`
→ `M = 250` samples, `mX` has 126 samples, non-zero at bin indices **2** (80 Hz) and **5** (200 Hz).

**Test case 2:** `fs = 48000 Hz`, `f1 = 300 Hz`, `f2 = 800 Hz`
→ `M = 480` samples, `mX` has 241 samples, non-zero at bin indices **3** (300 Hz) and **8** (800 Hz).

Generate each test signal as the sum of two cosines at `f1` and `f2`, then pass it to `minimize_energy_spread_dft()`.


In [ ]:
# E3 - 1.2: Compute and plot the two input signals proposed above, call the function minimize_energy_spread_dft(),
# and plot the output magnitude spectra

### Your code here



**E3 - 1.3: Conceptual questions (answer here)**

1. Explain in your own words why `M` must be the LCM of the two sinusoid periods. What goes wrong if you use a smaller value, such as the period of just one sinusoid?
2. From your plot of `mX` for test case 1, verify that the two non-zero bins correspond to 80 Hz and 200 Hz. Show the bin-index-to-Hz conversion formula and apply it to your results.
3. Suppose `f1` is not an integer factor of `fs` (e.g., `f1 = 75 Hz`, `fs = 10000 Hz`). Can you still find an integer `M` that yields exactly two non-zero bins? Explain why or why not.


## Part 2 – Symmetry properties of the DFT

The function `test_real_even()` should determine whether an input signal is real and even by inspecting the symmetry of its DFT, and return the result along with intermediate computations.

**Steps:**
1. Apply **zero-phase windowing** to `x` (no zero-padding — use `N = M`) to remove the phase offset caused by the signal's position in the frame.
2. Compute the `M`-point DFT of `dftbuffer`.
3. Check whether the result is real-valued (i.e., the imaginary part is negligible) using a tolerance of `1e-6`. If `imag(X[k]) ≈ 0` for all `k`, the signal is real and even.

**Inputs:** `x` — a signal of odd length `M`.

**Output:** tuple `(isRealEven, dftbuffer, X)` where:
- `isRealEven` (bool): `True` if `x` is real and even after zero-phase windowing
- `dftbuffer` (np.array): `M`-length zero-phase windowed version of `x`
- `X` (np.array): `M`-point DFT of `dftbuffer`

Use `np.abs(X.imag) < 1e-6` to test for zero imaginary part rather than comparing phase values directly.


In [ ]:
# E3 - 2.1: Complete the function test_real_even()

def test_real_even(x):
    """check if x is real and even using the symmetry properties of its DFT.
    Args:
        x (np.array): input signal of length M (M is odd)

    Returns:
        tuple including:
        isRealEven (boolean): True if input x is real and even, and False otherwise
        dftbuffer (np.array): M point zero phase windowed version of x
        X (np.array): M point DFT of dftbuffer

    """
    ### Your code here


Test cases for `test_real_even()`:

**Test case 1:** `x = np.array([2, 3, 4, 3, 2])` — real and even signal
```
(True,
 array([4., 3., 2., 2., 3.]),
 array([14.+0.j,  2.618+0.j,  0.382+0.j,  0.382+0.j,  2.618+0.j]))
```

**Test case 2:** `x = np.array([1, 2, 3, 4, 1, 2, 3])` — not even
```
(False,
 array([4., 1., 2., 3., 1., 2., 3.]),
 array([16.+0.j, 2.+0.69j, 2.+3.51j, 2.-1.08j, 2.+1.08j, 2.-3.51j, 2.-0.69j]))
```
*(values are approximate)*

For a more realistic case, use a symmetric Hann window:
```python
x = get_window('hann', 51, fftbins=False)
```
This is real and even. Plot `x` and the real/imaginary parts of its DFT spectrum `X`.


In [ ]:
# E3 - 2.2: Plot the input signal proposed above (window signal), call the function test_real_even(),
# and plot its output spectrum (real and imaginary)

### Your code here


**E3 - 2.3: Conceptual questions (answer here)**

1. Inspect the `dftbuffer` computed from `x = np.array([2, 3, 4, 3, 2])`. How do the sample values compare to the original `x`? What does zero-phase windowing do to the signal's position in the buffer?
2. For the Hann window input, look at the imaginary part of the output spectrum `X`. Is it exactly zero? What does this tell you about the symmetry of the Hann window?
3. Apply `test_real_even()` to `x = np.array([1, 2, 3, 2, 1])` (symmetric) and `x = np.array([1, 2, 3, 4, 5])` (asymmetric). Predict the result for each before running, then verify. Explain why one passes and the other fails.


## Part 3 – Suppress frequency components using the DFT model

The function `suppress_freq_dft_model()` should suppress all frequency components at or below 70 Hz using the DFT, and return the filtered time-domain signal.

**Steps:**
1. Apply a **Hamming window** of length `M` to `x`.
2. Use `dftAnal()` to obtain the magnitude spectrum `mX` (in dB) and phase spectrum `pX` of the windowed signal.
3. Set all values of `mX` at frequency bins `<= 70 Hz` to `-120` dB (i.e., effectively zero). Find the cutoff bin using `np.ceil(70.0 * N / fs)`.
4. Use `dftSynth()` to synthesize the filtered output signal, then **divide by the sum of the window** to undo the windowing gain.

**Inputs:**
- `x` (np.array): input signal of odd length `M`
- `fs` (float): sampling frequency in Hz
- `N` (int): FFT size

**Output:** filtered signal (np.array of length `N`).

> Note: this approach implements a rectangular frequency-domain cutoff, which introduces time-domain ringing (Gibbs phenomenon). It is illustrative, not a practical filtering method.


In [ ]:
# E3 - 3.1: Complete the function suppress_freq_dft_model()

def suppress_freq_dft_model(x, fs, N):
    """
    Args:
        x (np.array): input signal of length M (odd size)
        fs (float): sampling frequency (Hz)
        N (int): FFT size

    Returns:
       np.array: output signal with filtering (N samples long)
    """
    M = len(x)
    w = get_window('hamming', M)
    outputScaleFactor = sum(w)

    ### Your code here


Test case for the function `suppress_freq_dft_model()`:

_Test case 1:_ For an input signal with 40Hz, 100Hz, 200Hz, 1000Hz components, the output should only contain 100Hz, 200Hz and 1000Hz components.

_Test case 2:_ For an input signal with 23Hz, 36Hz, 230Hz, 900Hz, 2300Hz components, the output should only contain 230Hz, 900Hz and 2300Hz components.

To understand the effect of filtering, you can plot the magnitude spectra of the input and output signals superposed.

In [ ]:
# E3 - 3.2: Compute the input signals proposed above and plot their magnitude spectra (x-axis in Hz),
# call the function suppress_freq_dft_model(), and plot the magnitude spectra of the output signals

### Your code here


**E3 - 3.3: Conceptual questions (answer here)**

1. Why is the threshold set to $-120$ dB rather than simply setting the DFT coefficients to 0? What would happen if you used 0 directly in `dftSynth()`?
2. Overlay the magnitude spectra (in dB, x-axis in Hz) of the input and output signals. Describe what changed and verify that the suppressed frequencies are below the cutoff.
3. Look at the time-domain output near the beginning and end of the signal. Do you observe any artifacts? Why do these appear even though the filtering was done cleanly in the frequency domain? What property of the filter's impulse response is responsible?


## Part 4 – Window size, FFT size, and zero-padding

The function `zp_fft_size_expt()` should compute three magnitude spectra of the same input signal using different window sizes and FFT sizes, to illustrate the effects of these parameters on spectral resolution and interpolation.

Use a 512-sample input signal at `fs = 1000 Hz` with a Hamming analysis window. The three configurations are:

| Config | Window size | FFT size | Effect |
|--------|------------|----------|--------|
| 1 | 256 | 256 | baseline |
| 2 | 512 | 512 | longer window → better frequency resolution |
| 3 | 256 | 512 | zero-padding (256 zeros added) → interpolated spectrum |

Use `dftAnal()` to obtain the positive-half magnitude spectrum (in dB) for each configuration. Return the three spectra as a list.

**Input:** `x` (np.array, 512 samples)
**Output:** list of three magnitude spectra `[mX1, mX2, mX3]`


In [ ]:
# E3 - 4.1: Complete the function zp_fft_size_expt()

def zp_fft_size_expt(x, window_size=[256, 512, 256], FFT_size=[256, 512, 512]):
    """compute magnitude spectra of x with different window sizes and FFT sizes.

    Args:
        x (np.array): input signal (512 samples long)

    Returns:
        list with magnitude spectra (np.array)
    """

    ### Your code here


Test case for `zp_fft_size_expt()`:

Use `x = 0.2*np.cos(2*np.pi*200*n) + 0.2*np.cos(2*np.pi*400*n)` where `n = np.arange(512)/fs` and `fs = 1000 Hz`. Call with default arguments:

```python
mag_spectra = zp_fft_size_expt(x)
```

Plot all three spectra on a common frequency axis (Hz) with different colors. You should observe:
- `mag_spectra[2]` is an interpolated version of `mag_spectra[0]` (zero-padding interpolates the DFT)
- `mag_spectra[1]` has a **narrower mainlobe** than `mag_spectra[0]` (longer window → better frequency resolution)


In [ ]:
# E3 - 4.2: Generate the test signal above, call zp_fft_size_expt(), and plot the three magnitude
#            spectra on a common frequency axis with different colors. Add a legend.

### Your code here


**E3 - 4.3: Conceptual questions (answer here)**

1. Compare `mag_spectra[0]` (M=256, N=256) and `mag_spectra[2]` (M=256, N=512). What is the relationship between them? Measure the bin spacing (in Hz) for each and explain why they differ or agree despite having the same window.
2. Compare `mag_spectra[0]` (M=256) and `mag_spectra[1]` (M=512). Which one has a narrower mainlobe? What physical quantity determines mainlobe width, and why does a longer window help?
3. Zero-padding does not add new information about the signal. Given that, what exactly does it contribute to the spectrum representation? When would zero-padding be useful in practice?
4. For frequency estimation (finding the peak bin and converting to Hz), which of the three spectra gives the most accurate estimate? Justify your answer based on what you observe in the plots.
